# CH101 free hybrid quality strategies

This notebook runs each new strategy at most once: the low-memory PartCrafter part-level lane is preferred for CH101 semantic boundaries, followed by TRELLIS.2 and original TRELLIS fallbacks. All providers are guarded by GPU/VRAM/CUDA/license preflight, and every candidate uses the same refine, evaluate, score, strict visual QA, and ranking path. A failed strategy is never silently retried, and all Unity/Production gates remain locked.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

CHARACTER_CODE = 'CH101'
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
TOOLS_COMMIT_EXPECTED = os.environ.get('RE_CAMP_BLENDER_TOOLS_COMMIT', '')
ART_REPO_URL = 'https://github.com/siri2677/re-camp.git'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
TRELLIS_REPO_URL = 'https://github.com/microsoft/TRELLIS.git'
TRELLIS2_REPO_URL = 'https://github.com/microsoft/TRELLIS.2.git'
PARTCRAFTER_REPO_URL = 'https://github.com/wgsxm/PartCrafter.git'
SPAR3D_REPO_URL = 'https://github.com/Stability-AI/stable-point-aware-3d.git'
TRELLIS_COMMIT = '442aa1e1afb9014e80681d3bf604e8d728a86ee7'
TRELLIS16_COMMIT = '442aa1e1afb9014e80681d3bf604e8d728a86ee7'
TRELLIS2_COMMIT = '75fbf0183001ed9876c8dbb35de6b68552ee08bd'
PARTCRAFTER_COMMIT = '3d773bf02fad51c7ab31a5615573fec93b287b30'
SPAR3D_COMMIT = 'fdc311b16809e6a8adc2f5a3407ebb3db1a95bd1'
TRELLIS_STRATEGY_ID = 'TRELLIS_SINGLE_VIEW_V001'
TRELLIS16_STRATEGY_ID = 'TRELLIS_SINGLE_VIEW_16GB_V002'
TRELLIS2_STRATEGY_ID = 'TRELLIS2_SINGLE_VIEW_V001'
PARTCRAFTER_STRATEGY_ID = 'PARTCRAFTER_PART_LEVEL_V001'
SPAR3D_STRATEGY_ID = 'SPAR3D_SINGLE_VIEW_V001'
SEMANTIC_STRATEGY_ID = 'SEMANTIC_PROXY_REFERENCE_FITTED_V001'
UNIFIED_SEMANTIC_STRATEGY_ID = 'UNIFIED_SEMANTIC_AUTHORING_V002'
DETAIL_SEMANTIC_STRATEGY_ID = 'SEMANTIC_DETAIL_AUTHORING_V003'
RUNTIME_NAME = os.environ.get('RE_CAMP_RUNTIME', '').strip().lower()
if not RUNTIME_NAME:
    RUNTIME_NAME = 'kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/working').is_dir() else 'colab'
CONTENT_ROOT = Path('/kaggle/working' if RUNTIME_NAME == 'kaggle' else '/content')
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE / 'hybrid'
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
EVALUATION_DIR = OUTPUT_ROOT / 'evaluation'
SEMANTIC_DIR = OUTPUT_ROOT / 'semantic-proxy'
PARTCRAFTER_DIR = OUTPUT_ROOT / 'partcrafter-output'
SPAR3D_DIR = OUTPUT_ROOT / 'spar3d-output'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidates'
REVIEW_DIR = OUTPUT_ROOT / 'review'
GATES = {'sourceStatus': 'AI_GENERATED_CANDIDATE_NOT_PRODUCTION', 'gateB': 'PENDING_HUMAN_REVIEW', 'unityInputAllowed': False, 'productionPromotionAllowed': False}
print({'runtime': RUNTIME_NAME, 'partcrafterStrategy': PARTCRAFTER_STRATEGY_ID, 'spar3dStrategy': SPAR3D_STRATEGY_ID, 'trellisStrategy': TRELLIS_STRATEGY_ID, 'trellis16Strategy': TRELLIS16_STRATEGY_ID, 'semanticStrategy': SEMANTIC_STRATEGY_ID, **GATES})

In [ ]:
def run(command, *, cwd=None, check=True, env=None):
    command = [str(part) for part in command]
    print('RUN:', ' '.join(command))
    result = subprocess.run(command, cwd=cwd, env=env, check=False)
    if check and result.returncode:
        raise RuntimeError(f'command failed ({result.returncode}): {command}')
    return result

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
TOOLS_COMMIT = subprocess.check_output(['git', '-C', TOOLS_DIR, 'rev-parse', 'HEAD'], text=True).strip()
if TOOLS_COMMIT_EXPECTED: assert TOOLS_COMMIT == TOOLS_COMMIT_EXPECTED
if not (ART_DIR / '.git').is_dir():
    run(['git', 'clone', ART_REPO_URL, ART_DIR])
run(['git', '-C', ART_DIR, 'fetch', 'origin', ART_COMMIT])
run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py', '--art-root', ART_DIR, '--output-dir', REFERENCE_DIR, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
SEMANTIC_HANDOFF = OUTPUT_ROOT / 'semantic-reconstruction-inputs.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_semantic_reconstruction_handoff.py', '--art-root', ART_DIR, '--output', SEMANTIC_HANDOFF, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--character', CHARACTER_CODE])
print({'referenceManifest': str(REFERENCE_MANIFEST), 'semanticHandoff': str(SEMANTIC_HANDOFF), **GATES})

In [ ]:
# CPU Blender is only for the semantic proxy; it is never installed for a blocked TRELLIS path.
BLENDER_BIN = shutil.which('blender')
if not BLENDER_BIN and os.environ.get('RE_CAMP_INSTALL_CPU_BLENDER', '1') == '1':
    install = run(['apt-get', 'update', '-qq'], check=False)
    if install.returncode == 0:
        run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'], check=False)
    BLENDER_BIN = shutil.which('blender')
BLENDER_LAUNCHER = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
print({'blender': BLENDER_BIN or 'BLOCKED_BLENDER_AUTHORING_ENVIRONMENT', 'cpuSemanticProxyAllowed': bool(BLENDER_BIN), **GATES})

In [ ]:
# quality_progress_gate runs before any candidate execution. A rejected strategy cannot be repeated.
# BLOCKED_PROVIDER_PREFLIGHT and REGENERATE_REQUIRED are terminal recorded states for this run.
ORCHESTRATION_REPORT = OUTPUT_ROOT / 'hybrid-quality-orchestration.json'
orchestration_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'hybrid_quality_orchestrator.py', '--art-root', ART_DIR, '--output', ORCHESTRATION_REPORT, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--character', CHARACTER_CODE, '--score-dir', EVALUATION_DIR, '--history-record', TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-08-28-wonder3d-selection-root-cause-v074.json', '--history-record', TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-08-28-kaggle-hybrid-semantic-proxy-v077.json', '--history-record', TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-08-29-quality-progress-gate-v002.json', '--history-record', TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-08-29-local-blender-v003-review-v001.json', '--history-record', TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-08-31-kaggle-semantic-authoring-v002-review.json', '--history-record', TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-09-02-kaggle-partcrafter-v002-review.json']
run(orchestration_command)
orchestration = json.loads(ORCHESTRATION_REPORT.read_text(encoding='utf-8'))
assert orchestration['unityInputAllowed'] is False and orchestration['productionPromotionAllowed'] is False
TRELLIS_PLAN = orchestration['strategies'][TRELLIS_STRATEGY_ID]
TRELLIS16_PLAN = orchestration['strategies'][TRELLIS16_STRATEGY_ID]
TRELLIS2_PLAN = orchestration['strategies'][TRELLIS2_STRATEGY_ID]
PARTCRAFTER_PLAN = orchestration['strategies'][PARTCRAFTER_STRATEGY_ID]
SPAR3D_PLAN = orchestration['strategies'][SPAR3D_STRATEGY_ID]
SEMANTIC_PLAN = orchestration['strategies'][SEMANTIC_STRATEGY_ID]
UNIFIED_PLAN = orchestration['strategies'][UNIFIED_SEMANTIC_STRATEGY_ID]
DETAIL_PLAN = orchestration['strategies'][DETAIL_SEMANTIC_STRATEGY_ID]
PARTCRAFTER_RUN_ALLOWED = PARTCRAFTER_PLAN['runAllowed'] is True
SPAR3D_RUN_ALLOWED = SPAR3D_PLAN['runAllowed'] is True and not PARTCRAFTER_RUN_ALLOWED
TRELLIS2_RUN_ALLOWED = TRELLIS2_PLAN['runAllowed'] is True and not PARTCRAFTER_RUN_ALLOWED and not SPAR3D_RUN_ALLOWED
TRELLIS16_RUN_ALLOWED = TRELLIS16_PLAN['runAllowed'] is True and not PARTCRAFTER_RUN_ALLOWED and not SPAR3D_RUN_ALLOWED and not TRELLIS2_RUN_ALLOWED
TRELLIS_RUN_ALLOWED = TRELLIS_PLAN['runAllowed'] is True and not PARTCRAFTER_RUN_ALLOWED and not SPAR3D_RUN_ALLOWED and not TRELLIS2_RUN_ALLOWED and not TRELLIS16_RUN_ALLOWED
SEMANTIC_RUN_ALLOWED = SEMANTIC_PLAN['runAllowed'] is True and not PARTCRAFTER_RUN_ALLOWED and not SPAR3D_RUN_ALLOWED and not TRELLIS_RUN_ALLOWED and not TRELLIS16_RUN_ALLOWED and not TRELLIS2_RUN_ALLOWED
UNIFIED_RUN_ALLOWED = UNIFIED_PLAN['runAllowed'] is True and not PARTCRAFTER_RUN_ALLOWED and not SPAR3D_RUN_ALLOWED and not TRELLIS_RUN_ALLOWED and not TRELLIS16_RUN_ALLOWED and not TRELLIS2_RUN_ALLOWED
DETAIL_RUN_ALLOWED = DETAIL_PLAN['runAllowed'] is True and not PARTCRAFTER_RUN_ALLOWED and not SPAR3D_RUN_ALLOWED and not TRELLIS_RUN_ALLOWED and not TRELLIS16_RUN_ALLOWED and not TRELLIS2_RUN_ALLOWED
(OUTPUT_ROOT / 'trellis-preflight.json').write_text(json.dumps(TRELLIS_PLAN['preflight'], ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'trellis16-preflight.json').write_text(json.dumps(TRELLIS16_PLAN['preflight'], ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'trellis2-preflight.json').write_text(json.dumps(TRELLIS2_PLAN['preflight'], ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'partcrafter-preflight.json').write_text(json.dumps(PARTCRAFTER_PLAN['preflight'], ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'spar3d-preflight.json').write_text(json.dumps(SPAR3D_PLAN['preflight'], ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'partcrafterStatus': PARTCRAFTER_PLAN['status'], 'spar3dStatus': SPAR3D_PLAN['status'], 'trellisStatus': TRELLIS_PLAN['status'], 'trellis16Status': TRELLIS16_PLAN['status'], 'trellis2Status': TRELLIS2_PLAN['status'], 'semanticStatus': SEMANTIC_PLAN['status'], 'unifiedSemanticStatus': UNIFIED_PLAN['status'], 'detailSemanticStatus': DETAIL_PLAN['status'], 'selectedStrategies': orchestration['selectedStrategies'], **GATES})

In [ ]:
# CPU Blender authoring is fallback-only and executes at most once per strategy.
CANDIDATE_MANIFESTS = []
SPAR3D_REPORT = OUTPUT_ROOT / 'spar3d-one-shot-run-report.json'
SPAR3D_REPORT.write_text(json.dumps({'status': SPAR3D_PLAN['status'], 'strategyId': SPAR3D_STRATEGY_ID, 'preflight': SPAR3D_PLAN['preflight'], **GATES}, indent=2) + '\n', encoding='utf-8')
PARTCRAFTER_REPORT = OUTPUT_ROOT / 'partcrafter-one-shot-run-report.json'
PARTCRAFTER_REPORT.write_text(json.dumps({'status': PARTCRAFTER_PLAN['status'], 'strategyId': PARTCRAFTER_STRATEGY_ID, 'preflight': PARTCRAFTER_PLAN['preflight'], **GATES}, indent=2) + '\n', encoding='utf-8')
PARTCRAFTER_DIAGNOSIS = OUTPUT_ROOT / 'partcrafter-quality-diagnosis.json'
PARTCRAFTER_DIAGNOSIS_RECORD = TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-09-02-kaggle-partcrafter-v002-review.json'
if PARTCRAFTER_DIAGNOSIS_RECORD.is_file():
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'diagnose_partcrafter_review.py', '--record', PARTCRAFTER_DIAGNOSIS_RECORD, '--output', PARTCRAFTER_DIAGNOSIS])
PARTCRAFTER_REPAIR_INPUT = os.environ.get('RE_CAMP_PARTCRAFTER_REPAIR_BLEND', '').strip()
PARTCRAFTER_REPAIR_REPORT = OUTPUT_ROOT / 'partcrafter-stored-artifact-repair' / 'repair-report.json'
if PARTCRAFTER_REPAIR_INPUT:
    repair_input = Path(PARTCRAFTER_REPAIR_INPUT).resolve()
    if not repair_input.is_file():
        raise FileNotFoundError(f'RE_CAMP_PARTCRAFTER_REPAIR_BLEND missing: {repair_input}')
    repair_dir = PARTCRAFTER_REPAIR_REPORT.parent
    repair_blend = repair_dir / 'CH101_PARTCRAFTER_REPAIRED_NOT_PRODUCTION.blend'
    repair_glb = repair_dir / 'CH101_PARTCRAFTER_REPAIRED_NOT_PRODUCTION.glb'
    repair_result = run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'repair_partcrafter_review_candidate.py', '--', '--blend', repair_input, '--output-blend', repair_blend, '--output-glb', repair_glb, '--report', PARTCRAFTER_REPAIR_REPORT, '--character', CHARACTER_CODE, '--max-triangles', os.environ.get('RE_CAMP_PARTCRAFTER_MAX_TRIANGLES', '300000')], check=False)
    repair_payload = json.loads(PARTCRAFTER_REPAIR_REPORT.read_text(encoding='utf-8')) if PARTCRAFTER_REPAIR_REPORT.is_file() else {}
    if repair_result.returncode == 0 and repair_payload.get('status') == 'REVIEW_REPAIR_APPLIED':
        repaired_transport = Path(repair_payload['transportPath'])
        repaired_candidate_dir = CANDIDATE_DIR / 'partcrafter-repair'
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', repaired_transport, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', repaired_candidate_dir, '--provider', 'partcrafter', '--strategy-id', PARTCRAFTER_STRATEGY_ID, '--source-stage', 'PARTCRAFTER_STORED_ARTIFACT_REPAIR', '--candidate-label', 'REPAIR001', '--metadata-json', PARTCRAFTER_REPAIR_REPORT, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
        CANDIDATE_MANIFESTS.append(repaired_candidate_dir / 'candidate-manifest.json')
    else:
        print({'partcrafterStoredRepair': repair_payload.get('status', 'NO_REPAIR_REPORT'), 'returnCode': repair_result.returncode, **GATES})
def register_spar3d():
    SPAR3D_PROVIDER_DIR = CONTENT_ROOT / 'provider-SPAR3D'
    if not (SPAR3D_PROVIDER_DIR / '.git').is_dir():
        run(['git', 'clone', SPAR3D_REPO_URL, SPAR3D_PROVIDER_DIR])
    run(['git', '-C', SPAR3D_PROVIDER_DIR, 'fetch', '--depth', '1', 'origin', SPAR3D_COMMIT])
    run(['git', '-C', SPAR3D_PROVIDER_DIR, 'checkout', '--detach', SPAR3D_COMMIT])
    setup_command = os.environ.get('RE_CAMP_SPAR3D_SETUP_COMMAND', '').split()
    if not setup_command:
        SPAR3D_REPORT.write_text(json.dumps({'status': 'BLOCKED_PROVIDER_SETUP_UNVERIFIED', 'strategyId': SPAR3D_STRATEGY_ID, 'requiredEnv': 'RE_CAMP_SPAR3D_SETUP_COMMAND', **GATES}, indent=2) + '\n', encoding='utf-8')
        return
    setup_result = run(setup_command, cwd=SPAR3D_PROVIDER_DIR, check=False)
    if setup_result.returncode != 0:
        SPAR3D_REPORT.write_text(json.dumps({'status': 'SPAR3D_SETUP_FAILED', 'strategyId': SPAR3D_STRATEGY_ID, 'returnCode': setup_result.returncode, **GATES}, indent=2) + '\n', encoding='utf-8')
        return
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_spar3d_candidate.py', '--provider-repo', SPAR3D_PROVIDER_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', SPAR3D_DIR, '--preflight', OUTPUT_ROOT / 'spar3d-preflight.json', '--output-report', SPAR3D_REPORT, '--texture-resolution', os.environ.get('RE_CAMP_SPAR3D_TEXTURE_RESOLUTION', '1024'), '--target-count', os.environ.get('RE_CAMP_SPAR3D_TARGET_COUNT', '20000'), '--execute'], check=False)
    payload = json.loads(SPAR3D_REPORT.read_text(encoding='utf-8'))
    if payload.get('status') == 'SPAR3D_EXECUTED' and payload.get('meshOutputs'):
        spar3d_candidate_dir = CANDIDATE_DIR / 'spar3d'
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', payload['meshOutputs'][0], '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', spar3d_candidate_dir, '--provider', 'spar3d', '--strategy-id', SPAR3D_STRATEGY_ID, '--source-stage', 'SPAR3D_LOW_VRAM_RESEARCH', '--candidate-label', '001', '--metadata-json', SPAR3D_REPORT, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
        CANDIDATE_MANIFESTS.append(spar3d_candidate_dir / 'candidate-manifest.json')
def register_partcrafter():
    PARTCRAFTER_PROVIDER_DIR = CONTENT_ROOT / 'provider-PartCrafter'
    if not (PARTCRAFTER_PROVIDER_DIR / '.git').is_dir():
        run(['git', 'clone', PARTCRAFTER_REPO_URL, PARTCRAFTER_PROVIDER_DIR])
    run(['git', '-C', PARTCRAFTER_PROVIDER_DIR, 'fetch', '--depth', '1', 'origin', PARTCRAFTER_COMMIT])
    run(['git', '-C', PARTCRAFTER_PROVIDER_DIR, 'checkout', '--detach', PARTCRAFTER_COMMIT])
    setup_command = os.environ.get('RE_CAMP_PARTCRAFTER_SETUP_COMMAND', '').split()
    if not setup_command:
        PARTCRAFTER_REPORT.write_text(json.dumps({'status': 'BLOCKED_PROVIDER_SETUP_UNVERIFIED', 'strategyId': PARTCRAFTER_STRATEGY_ID, 'requiredEnv': 'RE_CAMP_PARTCRAFTER_SETUP_COMMAND', **GATES}, indent=2) + '\n', encoding='utf-8')
        return
    setup_result = run(setup_command, cwd=PARTCRAFTER_PROVIDER_DIR, check=False)
    if setup_result.returncode != 0:
        PARTCRAFTER_REPORT.write_text(json.dumps({'status': 'PARTCRAFTER_SETUP_FAILED', 'strategyId': PARTCRAFTER_STRATEGY_ID, 'returnCode': setup_result.returncode, **GATES}, indent=2) + '\n', encoding='utf-8')
        return
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_partcrafter_candidate.py', '--provider-repo', PARTCRAFTER_PROVIDER_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', PARTCRAFTER_DIR, '--preflight', OUTPUT_ROOT / 'partcrafter-preflight.json', '--output-report', PARTCRAFTER_REPORT, '--num-parts', os.environ.get('RE_CAMP_PARTCRAFTER_NUM_PARTS', '6'), '--num-tokens', os.environ.get('RE_CAMP_PARTCRAFTER_NUM_TOKENS', '1024'), '--num-inference-steps', os.environ.get('RE_CAMP_PARTCRAFTER_NUM_INFERENCE_STEPS', '50'), '--seed', '101001', '--execute'], check=False)
    payload = json.loads(PARTCRAFTER_REPORT.read_text(encoding='utf-8'))
    if payload.get('status') == 'PARTCRAFTER_EXECUTED' and payload.get('meshOutputs'):
        partcrafter_candidate_dir = CANDIDATE_DIR / 'partcrafter'
        register_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', payload['meshOutputs'][0], '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', partcrafter_candidate_dir, '--provider', 'partcrafter', '--strategy-id', PARTCRAFTER_STRATEGY_ID, '--source-stage', 'PARTCRAFTER_PART_LEVEL_RESEARCH', '--candidate-label', '001', '--metadata-json', PARTCRAFTER_REPORT, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE]
        for part_path in payload.get('partOutputs', []):
            register_command.extend(['--asset-file', part_path])
        if payload.get('providerManifest'):
            register_command.extend(['--asset-file', payload['providerManifest']])
        run(register_command)
        CANDIDATE_MANIFESTS.append(partcrafter_candidate_dir / 'candidate-manifest.json')
def register_semantic_v001():
    if not BLENDER_BIN:
        raise RuntimeError('BLOCKED_BLENDER_AUTHORING_ENVIRONMENT')
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'build_ch101_semantic_proxy.py', '--', '--output-dir', SEMANTIC_DIR, '--reference-report', SEMANTIC_HANDOFF, '--art-root', ART_DIR, '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--render'])
    report_path = SEMANTIC_DIR / 'semantic-proxy-report.json'
    payload = json.loads(report_path.read_text(encoding='utf-8'))
    assert payload['unityInputAllowed'] is False and payload['productionPromotionAllowed'] is False
    mesh_path = payload.get('mesh') or payload['glb']
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', mesh_path, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', CANDIDATE_DIR, '--provider', 'semanticProxy', '--strategy-id', SEMANTIC_STRATEGY_ID, '--source-stage', 'SEMANTIC_PROXY_REFERENCE_FITTED', '--candidate-label', '001', '--metadata-json', report_path, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
    CANDIDATE_MANIFESTS.append(CANDIDATE_DIR / 'candidate-manifest.json')
def register_unified_v002():
    if not BLENDER_BIN:
        raise RuntimeError('BLOCKED_BLENDER_AUTHORING_ENVIRONMENT')
    unified_dir = OUTPUT_ROOT / 'unified-semantic-authoring'
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'build_ch101_unified_semantic_mesh.py', '--', '--output-dir', unified_dir, '--reference-report', SEMANTIC_HANDOFF, '--art-root', ART_DIR, '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--render'])
    report_path = unified_dir / 'unified-semantic-authoring-report.json'
    payload = json.loads(report_path.read_text(encoding='utf-8'))
    assert payload['unityInputAllowed'] is False and payload['productionPromotionAllowed'] is False
    mesh_path = payload.get('mesh')
    if not mesh_path:
        raise RuntimeError('UNIFIED_SEMANTIC_MESH_MISSING')
    unified_candidate_dir = CANDIDATE_DIR / 'unified-semantic'
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', mesh_path, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', unified_candidate_dir, '--provider', 'blenderSemanticAuthoring', '--strategy-id', UNIFIED_SEMANTIC_STRATEGY_ID, '--source-stage', 'UNIFIED_SEMANTIC_AUTHORING', '--candidate-label', '002', '--metadata-json', report_path, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
    CANDIDATE_MANIFESTS.append(unified_candidate_dir / 'candidate-manifest.json')
def register_detail_v003():
    if not BLENDER_BIN:
        raise RuntimeError('BLOCKED_BLENDER_AUTHORING_ENVIRONMENT')
    detail_dir = OUTPUT_ROOT / 'semantic-detail-authoring'
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'build_ch101_semantic_detail_candidate.py', '--', '--output-dir', detail_dir, '--reference-report', SEMANTIC_HANDOFF, '--art-root', ART_DIR, '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--render'])
    report_path = detail_dir / 'semantic-detail-authoring-report.json'
    payload = json.loads(report_path.read_text(encoding='utf-8'))
    assert payload['unityInputAllowed'] is False and payload['productionPromotionAllowed'] is False
    mesh_path = payload.get('mesh')
    if not mesh_path:
        raise RuntimeError('SEMANTIC_DETAIL_MESH_MISSING')
    detail_candidate_dir = CANDIDATE_DIR / 'semantic-detail'
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', mesh_path, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', detail_candidate_dir, '--provider', 'blenderSemanticDetailAuthoring', '--strategy-id', DETAIL_SEMANTIC_STRATEGY_ID, '--source-stage', 'SEMANTIC_DETAIL_AUTHORING', '--candidate-label', '003', '--metadata-json', report_path, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
    CANDIDATE_MANIFESTS.append(detail_candidate_dir / 'candidate-manifest.json')
if PARTCRAFTER_RUN_ALLOWED:
    register_partcrafter()
elif SPAR3D_RUN_ALLOWED:
    register_spar3d()
elif SEMANTIC_RUN_ALLOWED:
    register_semantic_v001()
elif UNIFIED_RUN_ALLOWED and not TRELLIS_RUN_ALLOWED and not TRELLIS16_RUN_ALLOWED:
    register_unified_v002()
elif DETAIL_RUN_ALLOWED and not TRELLIS_RUN_ALLOWED and not TRELLIS16_RUN_ALLOWED:
    register_detail_v003()
else:
    PARTCRAFTER_REPORT.write_text(json.dumps({'status': PARTCRAFTER_PLAN['status'], 'strategyId': PARTCRAFTER_STRATEGY_ID, 'preflight': PARTCRAFTER_PLAN['preflight'], **GATES}, indent=2) + '\n', encoding='utf-8')
    print({'semanticProxy': 'SKIPPED', 'unifiedSemantic': 'DEFERRED_TO_TRELLIS_FALLBACK' if TRELLIS_RUN_ALLOWED else 'SKIPPED', 'semanticDetail': 'DEFERRED_TO_TRELLIS_FALLBACK' if TRELLIS_RUN_ALLOWED else 'SKIPPED', 'partcrafter': 'SKIPPED', 'spar3d': 'SKIPPED', 'reason': SEMANTIC_PLAN['status'], 'unifiedReason': UNIFIED_PLAN['status'], 'detailReason': DETAIL_PLAN['status'], **GATES})
print({'candidateManifests': [str(path) for path in CANDIDATE_MANIFESTS], **GATES})

In [ ]:
# TRELLIS.2, original TRELLIS 16GB fallback, and older TRELLIS each run at most once and share the same candidate list.
TRELLIS2_REPORT = OUTPUT_ROOT / 'trellis2-one-shot-run-report.json'
if TRELLIS2_RUN_ALLOWED:
    TRELLIS2_DIR = CONTENT_ROOT / 'provider-TRELLIS2'
    if not (TRELLIS2_DIR / '.git').is_dir():
        run(['git', 'clone', TRELLIS2_REPO_URL, TRELLIS2_DIR])
    run(['git', '-C', TRELLIS2_DIR, 'fetch', '--depth', '1', 'origin', TRELLIS2_COMMIT])
    run(['git', '-C', TRELLIS2_DIR, 'checkout', '--detach', TRELLIS2_COMMIT])
    setup_command = os.environ.get('RE_CAMP_TRELLIS2_SETUP_COMMAND', '').split()
    if setup_command:
        setup_result = run(setup_command, cwd=TRELLIS2_DIR, check=False)
        if setup_result.returncode != 0:
            TRELLIS2_REPORT.write_text(json.dumps({'status': 'TRELLIS2_SETUP_FAILED', 'strategyId': TRELLIS2_STRATEGY_ID, 'returnCode': setup_result.returncode, **GATES}, indent=2) + '\n', encoding='utf-8')
        else:
            run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_trellis2_candidate.py', '--provider-repo', TRELLIS2_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', OUTPUT_ROOT / 'trellis2-output', '--preflight', OUTPUT_ROOT / 'trellis2-preflight.json', '--output-report', TRELLIS2_REPORT, '--execute'], check=False)
    else:
        TRELLIS2_REPORT.write_text(json.dumps({'status': 'BLOCKED_PROVIDER_SETUP_UNVERIFIED', 'strategyId': TRELLIS2_STRATEGY_ID, 'requiredEnv': 'RE_CAMP_TRELLIS2_SETUP_COMMAND', **GATES}, indent=2) + '\n', encoding='utf-8')
else:
    TRELLIS2_REPORT.write_text(json.dumps({'status': TRELLIS2_PLAN['status'], 'strategyId': TRELLIS2_STRATEGY_ID, 'preflight': TRELLIS2_PLAN['preflight'], **GATES}, indent=2) + '\n', encoding='utf-8')
trellis2_result = json.loads(TRELLIS2_REPORT.read_text(encoding='utf-8'))
if trellis2_result.get('status') == 'TRELLIS2_EXECUTED' and trellis2_result.get('meshOutputs'):
    for index, mesh_path in enumerate(trellis2_result.get('meshOutputs', []), start=1):
        trellis2_candidate_dir = CANDIDATE_DIR / 'trellis2' / f'{index:03d}'
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', mesh_path, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', trellis2_candidate_dir, '--provider', 'trellis2', '--strategy-id', TRELLIS2_STRATEGY_ID, '--source-stage', 'TRELLIS2_SINGLE_VIEW_RESEARCH', '--candidate-label', f'{index:03d}', '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
        CANDIDATE_MANIFESTS.append(trellis2_candidate_dir / 'candidate-manifest.json')
trellis2_has_mesh = trellis2_result.get('status') == 'TRELLIS2_EXECUTED' and bool(trellis2_result.get('meshOutputs'))
TRELLIS16_REPORT = OUTPUT_ROOT / 'trellis16-one-shot-run-report.json'
if TRELLIS16_RUN_ALLOWED:
    TRELLIS16_DIR = CONTENT_ROOT / 'provider-TRELLIS16'
    if not (TRELLIS16_DIR / '.git').is_dir():
        run(['git', 'clone', TRELLIS_REPO_URL, TRELLIS16_DIR])
    run(['git', '-C', TRELLIS16_DIR, 'fetch', '--depth', '1', 'origin', TRELLIS16_COMMIT])
    run(['git', '-C', TRELLIS16_DIR, 'checkout', '--detach', TRELLIS16_COMMIT])
    trellis16_setup_command = os.environ.get('RE_CAMP_TRELLIS16_SETUP_COMMAND', '').split()
    if trellis16_setup_command:
        trellis16_setup_result = run(trellis16_setup_command, cwd=TRELLIS16_DIR, check=False)
        if trellis16_setup_result.returncode != 0:
            TRELLIS16_REPORT.write_text(json.dumps({'status': 'TRELLIS16_SETUP_FAILED', 'strategyId': TRELLIS16_STRATEGY_ID, 'returnCode': trellis16_setup_result.returncode, **GATES}, indent=2) + '\n', encoding='utf-8')
        else:
            run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_trellis16_candidate.py', '--provider-repo', TRELLIS16_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', OUTPUT_ROOT / 'trellis16-output', '--preflight', OUTPUT_ROOT / 'trellis16-preflight.json', '--output-report', TRELLIS16_REPORT, '--texture-size', '1024', '--simplify', '0.95', '--execute'], check=False)
    else:
        TRELLIS16_REPORT.write_text(json.dumps({'status': 'BLOCKED_PROVIDER_SETUP_UNVERIFIED', 'strategyId': TRELLIS16_STRATEGY_ID, 'requiredEnv': 'RE_CAMP_TRELLIS16_SETUP_COMMAND', **GATES}, indent=2) + '\n', encoding='utf-8')
else:
    TRELLIS16_REPORT.write_text(json.dumps({'status': TRELLIS16_PLAN['status'], 'strategyId': TRELLIS16_STRATEGY_ID, 'preflight': TRELLIS16_PLAN['preflight'], **GATES}, indent=2) + '\n', encoding='utf-8')
trellis16_result = json.loads(TRELLIS16_REPORT.read_text(encoding='utf-8'))
if trellis16_result.get('status') == 'TRELLIS16_EXECUTED' and trellis16_result.get('meshOutputs'):
    for index, mesh_path in enumerate(trellis16_result.get('meshOutputs', []), start=1):
        trellis16_candidate_dir = CANDIDATE_DIR / 'trellis16' / f'{index:03d}'
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', mesh_path, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', trellis16_candidate_dir, '--provider', 'trellis16', '--strategy-id', TRELLIS16_STRATEGY_ID, '--source-stage', 'TRELLIS_SINGLE_VIEW_16GB_RESEARCH', '--candidate-label', f'{index:03d}', '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
        CANDIDATE_MANIFESTS.append(trellis16_candidate_dir / 'candidate-manifest.json')
trellis16_has_mesh = trellis16_result.get('status') == 'TRELLIS16_EXECUTED' and bool(trellis16_result.get('meshOutputs'))
TRELLIS_REPORT = OUTPUT_ROOT / 'trellis-one-shot-run-report.json'
if TRELLIS_RUN_ALLOWED:
    TRELLIS_DIR = CONTENT_ROOT / 'provider-TRELLIS'
    if not (TRELLIS_DIR / '.git').is_dir():
        run(['git', 'clone', TRELLIS_REPO_URL, TRELLIS_DIR])
    run(['git', '-C', TRELLIS_DIR, 'fetch', '--depth', '1', 'origin', TRELLIS_COMMIT])
    run(['git', '-C', TRELLIS_DIR, 'checkout', '--detach', TRELLIS_COMMIT])
    trellis_command = os.environ.get('RE_CAMP_TRELLIS_COMMAND', '').split()
    if trellis_command:
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_trellis_candidate.py', '--provider-repo', TRELLIS_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', OUTPUT_ROOT / 'trellis-output', '--preflight', OUTPUT_ROOT / 'trellis-preflight.json', '--output-report', TRELLIS_REPORT, '--execute', '--'] + trellis_command, check=False)
    else:
        TRELLIS_REPORT.write_text(json.dumps({'status': 'BLOCKED_PROVIDER_ENTRYPOINT_UNVERIFIED', 'strategyId': TRELLIS_STRATEGY_ID, **GATES}, indent=2) + '\n', encoding='utf-8')
else:
    TRELLIS_REPORT.write_text(json.dumps({'status': TRELLIS_PLAN['status'], 'strategyId': TRELLIS_STRATEGY_ID, 'preflight': TRELLIS_PLAN['preflight'], **GATES}, indent=2) + '\n', encoding='utf-8')
trellis_result = json.loads(TRELLIS_REPORT.read_text(encoding='utf-8'))
if trellis_result.get('status') == 'TRELLIS_EXECUTED' and trellis_result.get('meshOutputs'):
    for index, mesh_path in enumerate(trellis_result.get('meshOutputs', []), start=1):
        trellis_candidate_dir = CANDIDATE_DIR / 'trellis' / f'{index:03d}'
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', mesh_path, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', trellis_candidate_dir, '--provider', 'trellis', '--strategy-id', TRELLIS_STRATEGY_ID, '--source-stage', 'TRELLIS_SINGLE_VIEW_RESEARCH', '--candidate-label', f'{index:03d}', '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
        CANDIDATE_MANIFESTS.append(trellis_candidate_dir / 'candidate-manifest.json')
trellis_has_mesh = trellis_result.get('status') == 'TRELLIS_EXECUTED' and bool(trellis_result.get('meshOutputs'))
if not trellis2_has_mesh and not trellis16_has_mesh and not trellis_has_mesh and not CANDIDATE_MANIFESTS:
    if SEMANTIC_PLAN['runAllowed'] is True:
        register_semantic_v001()
    elif UNIFIED_RUN_ALLOWED:
        register_unified_v002()
    elif DETAIL_RUN_ALLOWED:
        register_detail_v003()
print({'trellis2Status': trellis2_result.get('status'), 'trellis16Status': trellis16_result.get('status'), 'trellisStatus': trellis_result.get('status'), 'trellisCandidatesRegistered': len([path for path in CANDIDATE_MANIFESTS if 'trellis' in str(path).lower()]), 'fallbackCandidatesRegistered': len(CANDIDATE_MANIFESTS), **GATES})
# Every registered candidate uses the existing refine -> evaluate -> score -> strict visual QA path.
SCORE_REPORTS = []
for manifest_path in CANDIDATE_MANIFESTS:
    payload = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
    entry = payload['candidates'][0]
    candidate_id = entry['candidateId']
    candidate_output = EVALUATION_DIR / candidate_id
    candidate_output.mkdir(parents=True, exist_ok=True)
    refined_glb = candidate_output / f'{candidate_id}_refined.glb'
    refined_blend = candidate_output / f'{candidate_id}_refined_NOT_PRODUCTION.blend'
    refinement_report = candidate_output / 'refinement-report.json'
    candidate_provider = entry.get('provider', 'semanticProxy')
    candidate_strategy = entry.get('strategyId', SEMANTIC_STRATEGY_ID)
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py', '--', '--candidate', entry['modelPath'], '--output-glb', refined_glb, '--output-blend', refined_blend, '--report', refinement_report, '--provider', candidate_provider, '--attempt', '1', '--parent-sha256', entry['sha256'], '--material-mode', 'preserve'])
    refinement_payload = json.loads(refinement_report.read_text(encoding='utf-8'))
    evaluation_candidate = Path(refinement_payload.get('refinedTransportPath') or entry['modelPath'])
    evaluation_report = candidate_output / 'evaluation-report.json'
    normalized_blend = candidate_output / f'{candidate_id}_normalized_NOT_PRODUCTION.blend'
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py', '--', '--candidate', evaluation_candidate, '--candidate-id', candidate_id, '--strategy-id', candidate_strategy, '--output-dir', candidate_output, '--report', evaluation_report, '--normalized-blend', normalized_blend, '--integrity-blend', refined_blend])
    score_report = candidate_output / 'candidate-score.json'
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py', '--reference-manifest', REFERENCE_MANIFEST, '--evaluation-report', evaluation_report, '--candidate-manifest', manifest_path, '--output', score_report, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
    SCORE_REPORTS.append(score_report)
if SCORE_REPORTS:
    REVIEW_DIR.mkdir(parents=True, exist_ok=True)
    ASSISTED_REVIEW = REVIEW_DIR / 'assisted-visual-review.json'
    review_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'build_assisted_visual_review.py', '--output', ASSISTED_REVIEW, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE]
    for report_path in SCORE_REPORTS:
        review_command.extend(['--score-report', report_path])
    run(review_command)
    RANKING_MANIFEST = OUTPUT_ROOT / 'ranking-manifest.json'
    ranking_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'rank_candidates.py', '--output', RANKING_MANIFEST, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE, '--assisted-visual-review', ASSISTED_REVIEW]
    for report_path in SCORE_REPORTS:
        ranking_command.extend(['--score-report', report_path])
    run(ranking_command)
    ranking = json.loads(RANKING_MANIFEST.read_text(encoding='utf-8'))
    assert ranking['unityInputAllowed'] is False and ranking['productionPromotionAllowed'] is False
    print({'strictVisualQA': ranking.get('status'), 'selectedCandidate': ranking.get('selectedCandidate'), **GATES})
else:
    ranking = {'status': 'NO_CANDIDATE_STRATEGY_READY', **GATES}
    print(ranking)

In [ ]:
# The TRELLIS.2/TRELLIS one-shot results and any registered fallback meshes were handled before scoring.
print({'partcrafterResult': json.loads(PARTCRAFTER_REPORT.read_text(encoding='utf-8')), 'spar3dResult': json.loads(SPAR3D_REPORT.read_text(encoding='utf-8')), 'trellis2Result': json.loads(TRELLIS2_REPORT.read_text(encoding='utf-8')), 'trellis16Result': json.loads(TRELLIS16_REPORT.read_text(encoding='utf-8')), 'trellisResult': json.loads(TRELLIS_REPORT.read_text(encoding='utf-8')), 'unifiedSemanticPlan': UNIFIED_PLAN, 'detailSemanticPlan': DETAIL_PLAN, **GATES})

In [ ]:
execution_report = {
    'schemaVersion': 'ch101-hybrid-quality-execution-v001',
    'character': CHARACTER_CODE,
    'toolsCommit': TOOLS_COMMIT,
    'artCommit': ART_COMMIT,
    'trellisProviderCommit': TRELLIS_COMMIT,
    'trellis16ProviderCommit': TRELLIS16_COMMIT,
    'trellis2ProviderCommit': TRELLIS2_COMMIT,
    'partcrafterProviderCommit': PARTCRAFTER_COMMIT,
    'spar3dProviderCommit': SPAR3D_COMMIT,
    'trellis': TRELLIS_PLAN,
    'trellis16': TRELLIS16_PLAN,
    'trellis2': TRELLIS2_PLAN,
    'partcrafter': PARTCRAFTER_PLAN,
    'spar3d': SPAR3D_PLAN,
    'semanticProxy': SEMANTIC_PLAN,
    'unifiedSemanticAuthoring': UNIFIED_PLAN,
    'semanticDetailAuthoring': DETAIL_PLAN,
    'selectedStrategies': orchestration['selectedStrategies'],
    'partcrafterDiagnosis': str(PARTCRAFTER_DIAGNOSIS) if PARTCRAFTER_DIAGNOSIS.is_file() else None,
    'partcrafterStoredRepairReport': str(PARTCRAFTER_REPAIR_REPORT) if PARTCRAFTER_REPAIR_REPORT.is_file() else None,
    'spar3dRunReport': str(SPAR3D_REPORT) if SPAR3D_REPORT.is_file() else None,
    'candidateManifests': [str(path) for path in CANDIDATE_MANIFESTS],
    'scoreReports': [str(path) for path in SCORE_REPORTS],
    'rankingManifest': str(OUTPUT_ROOT / 'ranking-manifest.json') if SCORE_REPORTS else None,
    'status': ranking.get('status', 'NO_CANDIDATE_STRATEGY_READY'),
    **GATES,
}
EXECUTION_REPORT = OUTPUT_ROOT / 'hybrid-quality-execution-report.json'
EXECUTION_REPORT.write_text(json.dumps(execution_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
archive = Path(shutil.make_archive(str(CONTENT_ROOT / f're-camp-{CHARACTER_CODE}-hybrid-NOT-PRODUCTION'), 'zip', OUTPUT_ROOT))
print({'executionReport': str(EXECUTION_REPORT), 'archive': str(archive), **GATES})
if RUNTIME_NAME == 'colab':
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception:
        print('Browser download unavailable; preserve the archive before the session ends.')
else:
    print('Kaggle archive retained in the output workspace; download it from the output panel.')